In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DoubleType, IntegerType
from delta.tables import DeltaTable

spark.sql("USE CATALOG ecommerce_catalog")

bronze_df = spark.read.table("bronze.products_raw")
bronze_df.limit(10).display()

In [0]:
mrp_num = F.expr("try_cast(regexp_replace(cast(mrp as string), '[^0-9.\\\\-]', '') as double)")

In [0]:
# Use try_to_timestamp to handle unparseable dates gracefully (returns NULL instead of throwing error)
date_candidates = [
    F.expr("try_to_timestamp(launch_date, 'yyyy-MM-dd HH:mm:ss')"),
    F.expr("try_to_timestamp(launch_date, 'yyyy-MM-dd')"),
    F.expr("try_to_timestamp(launch_date, 'dd/MM/yyyy')"),
    F.expr("try_to_timestamp(launch_date, 'dd-MM-yyyy')"),
    F.expr("try_to_timestamp(launch_date, 'MM/dd/yyyy')"),
    F.expr("try_to_timestamp(launch_date, 'yyyy/MM/dd HH:mm:ss')"),
    F.expr("try_to_timestamp(launch_date, 'dd MM yyyy')"),
    F.expr("try_to_timestamp(launch_date, 'dd MMM yyyy')"),
]
launch_date_clean = F.coalesce(*date_candidates)

In [0]:
category_clean = F.initcap(F.trim(F.col("category")))
brand_clean = F.trim(F.col("brand"))

In [0]:
#standardize boolean
is_organic_clean = (
    F.when(F.lower(F.col("is_organic").cast("string")).isin("y", "yes", "l", "true"), True)
    .when(F.lower(F.col("is_organic").cast("string")).isin("n", "no", "0", "false"), False)
    .otherwise(None)
)

silver_stage = (
    bronze_df.withColumn("mrp_clean", F.when(mrp_num > 0, mrp_num))
    .withColumn("launch_date_clean", launch_date_clean)
    
)

In [0]:
silver_stage = (
    bronze_df.withColumn("mrp_clean", F.when(mrp_num > 0, mrp_num))
    .withColumn("launch_date_clean", launch_date_clean)
    .withColumn("category_clean", category_clean)
    .withColumn("brand_clean", brand_clean)
    .withColumn("is_organic_clean", is_organic_clean)
    .withColumn("discount_percent_clean",
                F.when(F.col("discount_percent").cast(IntegerType()).between(0, 90),
                       F.col("discount_percent").cast(IntegerType())))
    .withColumn("stock_quantity_clean",
                F.when(F.col("stock_quantity").cast(IntegerType()).between(0, 50000),
                       F.col("stock_quantity").cast(IntegerType())))
    .withColumn("last_modified", F.col("updated_at"))
    .withColumn("cdc_operation", F.lit("I"))
)

In [0]:
#Deduplication Window

w = Window.partitionBy("product_id").orderBy(F.col("last_modified").desc())

#applying the window to dedupe

silver_dedup = silver_stage.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")


In [0]:
final_cols = [
    "product_id", "product_name", "brand_clean", "category_clean", "sub_category", "mrp_clean", "discount_percent_clean", "selling_price", "stock_quantity_clean", "rating", "rating_count", "seller_name", "warehouse_city", "is_cruelty_free", "is_organic_clean", "weight_grams", "launch_date_clean", "status", "source_system", "last_modified", "cdc_operation"
]

silver_new = silver_dedup.select(*final_cols)\
    .toDF(*[c.replace("_clean", "") for c in final_cols])

In [0]:
#Merge into silver SCD Type1
if not spark.catalog.tableExists("silver.products"):
    (silver_new.filter("cdc_operation != 'D'").drop("cdc_operation")
     .write.format("delta").saveAsTable("silver.products"))
else:
    target = DeltaTable.forName(spark, "silver.products")
    (
        target.alias("t")
        .merge(silver_new.alias("s"), "t.product_id = s.product_id")
        .whenMatchedDelete(condition="s.cdc_operation = 'D'")
        .whenMatchedUpdate(
            condition="s.cdc_operation != 'D'",
            set={
                "product_name": "s.product_name",
                "brand": "s.brand",
                "category": "s.category",
                "sub_category": "s.sub_category",
                "mrp": "s.mrp",
                "discount_percent": "s.discount_percent",
                "selling_price": "s.selling_price",
                "stock_quantity": "s.stock_quantity",
                "rating": "s.rating",
                "rating_count": "s.rating_count",
                "seller_name": "s.seller_name",
                "warehouse_city": "s.warehouse_city",
                "is_cruelty_free": "s.is_cruelty_free",
                "is_organic": "s.is_organic",
                "weight_grams": "s.weight_grams",
                "launch_date": "s.launch_date",
                "status": "s.status",
                "source_system": "s.source_system",
                "updated_at": "s.last_modified"
            }
        )
        .whenNotMatchedInsert(
            condition="s.cdc_operation != 'D'",
            values={
                "product_id": "s.product_id",
                "product_name": "s.product_name",
                "brand": "s.brand",
                "category": "s.category",
                "sub_category": "s.sub_category",
                "mrp": "s.mrp",
                "discount_percent": "s.discount_percent",
                "selling_price": "s.selling_price",
                "stock_quantity": "s.stock_quantity",
                "rating": "s.rating",
                "rating_count": "s.rating_count",
                "seller_name": "s.seller_name",
                "warehouse_city": "s.warehouse_city",
                "is_cruelty_free": "s.is_cruelty_free",
                "is_organic": "s.is_organic",
                "weight_grams": "s.weight_grams",
                "launch_date": "s.launch_date",
                "status": "s.status",
                "source_system": "s.source_system",
                "created_at": "s.last_modified",
                "updated_at": "s.last_modified"
            }
        )
        .execute()
    )
    
print("silver.products row count: ", spark.table("silver.products").count())



In [0]:
#Reusable quarantine pattern
orphans = silver_new.join(spark.table("silver.products").select("product_id"), "product_id", "left_anti")
#Write orphans to quarantine
orphans.write.format("delta").mode("append").saveAsTable("silver.orders_rejected")
silver_new = silver_new.join(orphans.select("order_id"), "order_id", "left_anti")

#Email validity check (regex), routed to the same quarantine pattern or nulled out
valid_email = F.col("customer_email").rlike(r"^[\w\.-]+@[\w\.-]+\.\w+$")
silver_new = silver_new.withColumn("customer_email", F.when(valid_email, F.trim(F.col("customer_email"))))
